<a href="https://colab.research.google.com/github/taehi5/Esaa_YB/blob/11%EC%A3%BC%EC%B0%A8---%EA%B3%BC%EC%A0%9C/YB_1113_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C_%ED%9A%8C%EA%B7%801.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. 단순 선형 회귀 분석**
- 전복의 나이를 예측하는 선형회귀모델을 생성하세요.
- 전복의 ‘성별’, ‘키’, ‘지름’, ‘높이’, ‘전체무게’, ‘몸통무게’, ‘내장무게’, ‘껍질무게’를 이용해 ‘껍질의 고리 수’를 예측한 뒤, **예측된 ‘껍질의 고리 수’에 1.5를 더하면 전복의 나이**가 됩니다.

In [1]:
# 기본 모듈 불러오기
import numpy as np
import pandas as pd
from IPython.display import display

**1) 데이터 load 및 변형**

In [2]:
# 풀이 노트북과 같은 폴더에 abalone.csv를 둡니다.
# Colab에서는 왼쪽 파일 탭에 abalone.csv를 업로드한 뒤 실행합니다.
data = pd.read_csv("/content/abalone.csv")
display(data.head())
print('데이터 크기:', data.shape)

# 성별 M은 Male, F는 Female, I는 Infant
for label in "MFI":
    data[label] = data["Sex"] == label
data.drop('Sex', axis=1, inplace=True)

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


데이터 크기: (4177, 9)


**2) X, y 선택**
: y는 Rings열, X는 Rings열을 제외한 나머지를 선택하되 전부 실수가 되도록 한다.

In [3]:
# Rings를 정답으로 분리하고, 모든 피처를 실수형으로 변환
y = data['Rings'].astype(float)
X = data.drop('Rings', axis=1).astype(float)
print('X 크기:', X.shape, 'y 크기:', y.shape)
display(X.head())

X 크기: (4177, 10) y 크기: (4177,)


,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,M,F,I
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,1.0,0.0,0.0
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,1.0,0.0,0.0
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,0.0,1.0,0.0
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,1.0,0.0,0.0
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,0.0,0.0,1.0


 **3) train/test set 분리**

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
# 재현 가능한 7:3 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=156
)
print('학습 데이터:', X_train.shape, '테스트 데이터:', X_test.shape)

학습 데이터: (2923, 10) 테스트 데이터: (1254, 10)


**4) 선형회귀모델 생성, 모델 예측치 구하기**

In [6]:
from sklearn.linear_model import LinearRegression

In [7]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [8]:
# 고리 수 예측
y_pred = lr.predict(X_test)

# 고리 수 + 1.5 = 전복의 추정 나이
age_pred = y_pred + 1.5
age_true = y_test + 1.5
prediction_df = pd.DataFrame({
    '실제 고리 수': y_test,
    '예측 고리 수': y_pred,
    '실제 나이(고리 수+1.5)': age_true,
    '예측 나이': age_pred
})
display(prediction_df.head(10))

,실제 고리 수,예측 고리 수,실제 나이(고리 수+1.5),예측 나이
2604,10.0,11.436056,11.5,12.936056
946,8.0,8.816801,9.5,10.316801
1122,9.0,8.559434,10.5,10.059434
542,15.0,9.630772,16.5,11.130772
3600,4.0,4.680213,5.5,6.180213
2485,8.0,7.147926,9.5,8.647926
1191,10.0,9.714780,11.5,11.214780
980,9.0,8.592959,10.5,10.092959
4108,7.0,7.322262,8.5,8.822262
1829,6.0,7.127594,7.5,8.627594


**5) 모델 평가: MSE, RMSE, R2 score, corr 구하기**

In [9]:
from sklearn.metrics import mean_squared_error, r2_score

- MSE, RMSE

In [10]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print('MSE: {:.4f}'.format(mse))
print('RMSE: {:.4f}'.format(rmse))

MSE: 5.1926
RMSE: 2.2787


- R2 score

In [11]:
r2 = r2_score(y_test, y_pred)
print('R2 score: {:.4f}'.format(r2))

R2 score: 0.4863


- 회귀 절편값

In [12]:
print('회귀 절편 값:', lr.intercept_)

회귀 절편 값: 3.248395545002505


- 회귀 계수 값

In [13]:
coeff = pd.Series(lr.coef_, index=X_train.columns, name='회귀 계수')
display(coeff)

,회귀 계수
Length,-2.673233
Diameter,11.159886
Height,24.245150
Whole weight,9.576390
Shucked weight,-20.226692
Viscera weight,-11.627877
Shell weight,6.587049
M,0.265019
F,0.242386
I,-0.507404


- 상관계수

Hint: corr 함수 이용.

In [14]:
# 실제 고리 수와 예측 고리 수 사이의 피어슨 상관계수
corr_df = pd.DataFrame({'actual': y_test, 'predicted': y_pred})
display(corr_df.corr())
corr = corr_df['actual'].corr(corr_df['predicted'])
print('실제값과 예측값의 상관계수: {:.4f}'.format(corr))

# 피처를 포함한 전체 변수 간 상관계수도 확인
display(data.astype(float).corr())

,actual,predicted
actual,1.000000,0.701207
predicted,0.701207,1.000000


실제값과 예측값의 상관계수: 0.7012


,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings,M,F,I
Length,1.000000,0.986812,0.827554,0.925261,0.897914,0.903018,0.897706,0.556720,0.236543,0.309666,-0.551465
Diameter,0.986812,1.000000,0.833684,0.925452,0.893162,0.899724,0.905330,0.574660,0.240376,0.318626,-0.564315
Height,0.827554,0.833684,1.000000,0.819221,0.774972,0.798319,0.817338,0.557467,0.215459,0.298421,-0.518552
Whole weight,0.925261,0.925452,0.819221,1.000000,0.969405,0.966375,0.955355,0.540390,0.252038,0.299741,-0.557592
Shucked weight,0.897914,0.893162,0.774972,0.969405,1.000000,0.931961,0.882617,0.420884,0.251793,0.263991,-0.521842
Viscera weight,0.903018,0.899724,0.798319,0.966375,0.931961,1.000000,0.907656,0.503819,0.242194,0.308444,-0.556081
Shell weight,0.897706,0.905330,0.817338,0.955355,0.882617,0.907656,1.000000,0.627574,0.235391,0.306319,-0.546953
Rings,0.556720,0.574660,0.557467,0.540390,0.420884,0.503819,0.627574,1.000000,0.181831,0.250279,-0.436063
M,0.236543,0.240376,0.215459,0.252038,0.251793,0.242194,0.235391,0.181831,1.000000,-0.512528,-0.522541
F,0.309666,0.318626,0.298421,0.299741,0.263991,0.308444,0.306319,0.250279,-0.512528,1.000000,-0.464298


# **2. Polynomial features**

In [15]:
from sklearn.preprocessing import PolynomialFeatures

In [16]:
# 임의 데이터 생성

X = np.arange(6).reshape(3, 2)

df =  pd.DataFrame(X)
df.columns = ['x_1','x_2']
df

,x_1,x_2
0,0,1
1,2,3
2,4,5


In [17]:
# 2차 이하 다항항 생성 (기본 include_bias=True: 상수항 1 포함)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(df)
df_poly = pd.DataFrame(X_poly, index=df.index)
display(df_poly)

,0,1,2,3,4,5
0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.0,2.0,3.0,4.0,6.0,9.0
2,1.0,4.0,5.0,16.0,20.0,25.0


In [18]:
df_poly.columns = ['1', 'x1', 'x2', 'x1^2', 'x1*x2', 'x2^2']
display(df_poly)

,1,x1,x2,x1^2,x1*x2,x2^2
0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.0,2.0,3.0,4.0,6.0,9.0
2,1.0,4.0,5.0,16.0,20.0,25.0
